<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Lab: Structure Unstructured Restaurant Data with an LLM**


Estimated time needed: **45** minutes


## **Scenario**


### Background

You are a Data Engineer at a leading AI startup building a next-generation *Connoisseur Companion*. Unlike traditional recommendation systems that rely on coarse 1–5 star ratings, the organization aims to understand the **why** behind user preferences, capturing the vibe of a restaurant, dietary considerations, and standout **“hero dishes”** hidden inside unstructured, multimodal data.

### The challenge

The organization has acquired a large, messy dataset spanning multiple modalities:

* **Restaurant descriptions** and **user reviews** (raw TXT)
* **Food recipes** with structured metadata and images (JSON + JPEG)
* **User restaurant visit histories** (JSON with URLs)

However, this data is not immediately usable:

1. **Unstructured text** lacks a consistent schema, making search and retrieval inefficient.
2. **Images and URLs** are opaque to traditional data pipelines and must be transformed into searchable, quantitative representations.
3. The data spans multiple modalities with no unified representation.

To build a high-quality, explainable recommendation engine, your first task is to transform this **heterogeneous, multimodal data** into a **structured knowledge base**. This includes extracting semantic signals from **text**, generating representations for **images** and **URLs**, and organizing everything into a coherent, query-friendly JSON format.


## **Objectives**


In this lab, you will write a Python program that will:

* Load raw restaurant descriptions and review data from unstructured TXT files
* Use a multimodal/LLM-based pipeline to extract structured attributes (e.g., cuisine type, ambiance, dietary options, and signature dishes)
* Convert the extracted information into a well-defined JSON schema suitable for indexing and search


## **Important: About the lab environment**


Please be aware that sessions for this lab environment are not persisted. Every time you connect to this lab, a new environment is created for you. Any data you may have saved in the earlier session would get lost. Plan to complete these labs in a single session, to avoid losing your data.


## **Screenshot requirement for this lab**


You will be prompted to take a screenshot and save it on your own device. You will need this screenshot either to answer graded quiz questions or to upload as your submission for the Final Project at the end of this course. You can use various free screen-grabbing tools or your operating system's shortcut keys to do this (for example, `Alt+PrintScreen` on Windows and `Command+shift+4` on Mac).

**Note**: The screenshot can be saved with either the **.jpg** or **.png** extension.


----


## **Set up the lab environment**


For this lab, you will be using the following libraries:

* [`numpy`](https://numpy.org/) for numerical operations and handling array-based data during preprocessing and analysis

* [`matplotlib`](https://matplotlib.org/) for basic data visualization and plotting, useful for inspecting distributions or intermediate results

* [`json`](https://docs.python.org/3/library/json.html) for parsing, constructing, and serializing structured JSON representations extracted from unstructured text

* **IBM Watsonx AI SDK** (`ibm-watsonx-ai`) for interacting with foundation models hosted on IBM watsonx.ai:

  * `Credentials` to securely authenticate with the watsonx.ai service
  * `ModelInference` to invoke foundation models for text understanding and information extraction
  * `GenTextParamsMetaNames` to configure text generation and extraction parameters
  * `ModelTypes` and `DecodingMethods` to select the appropriate model and control inference behavior

These libraries together enable you to transform raw restaurant description text into structured, machine-readable knowledge using GenAI-powered workflows.


### Install required libraries

Run the following code block to install all required libraries:


### Import required libraries

It is recommended to import all required libraries in one place (here):


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import json
import os

# # IBM WatsonX imports
# from ibm_watsonx_ai import Credentials
# from ibm_watsonx_ai.foundation_models import ModelInference
# from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
# from ibm_watsonx_ai.foundation_models.utils.enums import (
#     ModelTypes,
#     DecodingMethods,
# )

# Libraries and codes to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

### Fetch the data file


Run the following code to fetch the restaurant description text file:


In [3]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/1r_mM6ZPYNxcFv65QkzubA/California-Culinary-Map.txt

--2026-09-04 17:17:00--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/1r_mM6ZPYNxcFv65QkzubA/California-Culinary-Map.txt
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 74679 (73K) [text/plain]
Saving to: ‘California-Culinary-Map.txt.3’

California-Culinary 100%[===================>]  72,93K   325KB/s    in 0,2s    

2026-09-04 17:17:01 (325 KB/s) - ‘California-Culinary-Map.txt.3’ saved [74679/74679]



----


## **Exercise 1: Load the data and display the texts**


Great! You now have the **California-Culinary-Map.txt** file in your folder. Before diving into the data, it is important to load and explore it first to gain a high-level understanding of its structure and content. This exercise gives you the opportunity to inspect the raw data and prepare it for transformation into a more structured, machine-accessible format.


### Step 1: Load the data

In Step 1, you will load the text data file and explore the restaurant contents. Make sure you use the correct file path that links to the file California-Culinary-Map.txt. 


In [4]:
### Your Code Here:
### 1.1: Define the file_path to the text file
file_path = "../data/California-Culinary-Map.txt"

### 1.2: Open the text file
with open(file_path, 'r') as file:
    data = file.read()

### 1.3: Print the first 100 characters of the restaurant data
print(data[:100])

### The Culinary Map of California

**The Gilded Artichoke** brings a **bohemian chic** energy to th


### Step 2: Split the restaurant paragraphs into a Python list

In the previous step, you may have noticed that the text file consists of multiple paragraphs, each describing a single restaurant. In Step 2, you will split these paragraphs into a list, allowing you to work with and manage each restaurant’s data more effectively.


In [5]:
### Your Code Here:
### 2.1: Split the restaurant paragraphs into list (hint: use .split('\n\n')
restaurant_list = data.split('\n\n')

### 2.2: Since the first item is the dataset name, we remove it
restaurant_list = restaurant_list[1:]

### 2.3: Print out the number of restaurants we have (hint: use the len() function)
print(len(restaurant_list))

### 2.4: Print out the first item to have a closer look at the content
print(restaurant_list[0])

210
**The Gilded Artichoke** brings a **bohemian chic** energy to the hills of **Silver Lake**, operating as an **upscale bistro** that prioritizes **Farm-to-Table Californian** ingredients. The space feels like a high-end greenhouse with its reclaimed wood and floor-to-ceiling windows, perfectly complementing the **4.5/5** rating earned by its lavender-rubbed roasted chicken and delicate heirloom tomato tarts.  Price range: $$$$


----


## **Exercise 2: Define the LLM**


You may have observed that each restaurant description includes the following key attributes: 
1. Restaurant name
2. Location
3. Restaurant type
4. Food style
5. Rating
6. Price range
7. Signature dishes
8. Specialties
9. Shortcomings

This information is essential for uniquely identifying and describing a restaurant.

In this lab, you will use a large language model (LLM) to organize these attributes into a structured JSON format. As a first step, you will define the base LLM that will extract and structure the information from the unstructured text.


### Step 1: Implement the LLM function


You will use **IBM Granite 4H Small** as the base LLM for this lab, as it provides strong text understanding capabilities while remaining cost-efficient:
### I will use local ollama models


In [6]:
! ollama list

]11;?\NAME                        ID              SIZE      MODIFIED     
llava:latest                8dd30f6b0cb1    4.7 GB    10 days ago     
llama3.2:latest             a80c4f17acd5    2.0 GB    10 days ago     
llama3.2:3b                 a80c4f17acd5    2.0 GB    10 days ago     
gemma4:latest               c6eb396dbd59    9.6 GB    4 months ago    
qwen3.5:27b                 7653528ba5cb    17 GB     5 months ago    
qwen35-uncensored:latest    443fb05972d8    6.7 GB    5 months ago    
deepseek-r1:latest          6995872bfe4c    5.2 GB    5 months ago    
deepseek-r1:8b              6995872bfe4c    5.2 GB    5 months ago    
codestral:latest            0898a8b286d5    12 GB     5 months ago    
devstral-2:latest           524a6607f0f5    74 GB     5 months ago    
qwen3-coder-next:latest     ca06e9e4087c    51 GB     6 months ago    
deepseek-coder:33b          acec7c0b0fd9    18 GB     6 months ago    
qwen3-coder:latest          06c1097efce0    18 GB     6 months ago    


In [7]:
from langchain_ollama import ChatOllama


def llm_model(system_msg, prompt_txt):
    model_id = "llava:latest"

    # Define the model
    model = ChatOllama(
        model=model_id,
        temperature=0
    )

    # Define messages
    messages = [
        ("system", system_msg),
        ("user", prompt_txt)
    ]

    # Get response
    response = model.invoke(messages)

    # Extract text
    output_text = response.content

    return output_text

### Step 2: Test your llm_model()


Test your LLM function with the following inputs:


In [8]:
import time

def safe_llm_call(system_msg, prompt_txt, retries=3):
    for i in range(retries):
        try:
            return llm_model(system_msg, prompt_txt)
        except Exception:
            time.sleep(2)
    return "Failed after retries"

In [9]:
system_msg = "You are a helpful assistant."
prompt_txt = "Which place is warmer in winter? Hawaii or Greenland?"
print(safe_llm_call(system_msg, prompt_txt))

 Hawaii is warmer in winter compared to Greenland. Hawaii is a tropical location with a warm climate year-round, while Greenland is a cold, icy region located near the North Pole. In winter, Greenland experiences freezing temperatures and snow, while Hawaii has a more temperate climate with mild temperatures and occasional rain. 


----


## **Exercise 3: Prompt Engineering**


To make an LLM behave reliably and produce outputs that meet your requirements, careful prompt design is essential. Here is a recap of key prompt engineering techniques:

* **Clear and specific instructions**: Explicitly state the task, expected output format, and constraints to reduce ambiguity.
* **Structured output guidance**: Provide schemas, templates, or examples (e.g., JSON formats) to encourage consistent, machine-readable responses.
* **Few-shot prompting**: Include representative input–output examples to guide the model toward the desired behavior.
* **Role prompting**: Assign the model a specific role (e.g., “You are a data extraction assistant”) to shape its reasoning and tone.
* **Constraint-based prompting**: Define what the model should and should not do to improve precision and reliability.
* **Iterative refinement**: Evaluate outputs and progressively refine prompts to improve performance over time.

This exercise focuses on one-shot prompting, a special case of few-shot prompting. You will define a function that generates the prompt templates.


### Step 1: Define the template


In **Step 1**, you will implement a function that generates prompts for the LLM, given an input restaurant description paragraph.

You will apply the **one-shot prompting** technique, which includes a single example of the desired output within the prompt to guide the model’s response.

An example output is provided, corresponding to the second restaurant paragraph in your `restaurant_list` variable. Using this example, you will design a prompt template that enables the model to consistently transform *any* input restaurant description into the required structured JSON format.

**Note:** For the price range field, instruct the LLM to convert dollar signs (e.g., \\\\\\\\\\\\\\\\\\\\\\\\$, \\$\\$, $$$) into an integer representing the number of dollar symbols.


# Your prompt should explicitly define the JSON schema, tell the model to extract only information present in the description, and require valid JSON. 

In [10]:
restaurant_list[1]

'Down in **Santa Monica**, **Mar de Cortez** serves as a **sun-drenched**, **casual taqueria** specializing in **Baja-style seafood**. With a **4.2/5** rating, it captures the salt-air energy of the coast through its signature beer-battered snapper tacos and zesty octopus ceviche, making it a premier spot for open-air dining near the pier. Price range: $'

In [11]:
EXAMPLE_RESTAURANT_PARAGRAPH = restaurant_list[1] #use the second restaurant paragraph as the example
EXAMPLE_OUTPUT = """
    {{
    "name": "Mar de Cortez",
    "location": "Santa Monica",
    "type": "casual taqueria",
    "food_style": "Baja-style seafood",
    "rating": 4.2,
    "price_range": 1,
    "signatures": [
        "beer-battered snapper tacos",
        "zesty octopus ceviche"
    ],
    "vibe": "salt-air energy",
    "environment": "a premier sun-drenched spot for open-air dining near the pier."
    "shortcomings": []
    }}
"""

def restaurant_data_structure_prompt_generation(restaurant_paragraph):

    base_system_msg = """
You are a restaurant information extraction assistant.

Your task is to extract structured information from a restaurant description
and return ONLY valid JSON.

Extract the following fields:

- name: restaurant name
- location: city or location
- type: type category of restaurant
- food_style: cuisine or food style
- rating: numerical rating if explicitly mentioned
- price_range: numerical price category if explicitly mentioned
- signatures: list of notable/signature dishes
- vibe: short description of the restaurant's atmosphere or character
- environment: description of the physical dining environment
- shortcomings: list of explicitly mentioned disadvantages, weaknesses, or drawbacks

Rules:
1. Return ONLY JSON. Do not include explanations, markdown, or code fences.
2. Do not invent information that is not present in the description.
3. If a field is not mentioned, use null.
4. If no signature dishes are mentioned, use an empty list [].
5. If no shortcomings are mentioned, use an empty list [].
6. Keep the extracted information concise.
7. The JSON must be syntactically valid.
"""

    base_user_prompt = f"""
Task:
Extract the restaurant information from the description below and convert it
into the exact JSON structure shown in the example.

Restaurant description:
{restaurant_paragraph}

Example input:
{EXAMPLE_RESTAURANT_PARAGRAPH}

Example output:
{EXAMPLE_OUTPUT}

Now extract the information from the restaurant description above.

Return ONLY the JSON object.
"""

    return base_system_msg, base_user_prompt


def clean_json_response(response):
    response = response.strip()

    if response.startswith("```json"):
        response = response[len("```json"):]

    elif response.startswith("```"):
        response = response[len("```"):]

    if response.endswith("```"):
        response = response[:-3]

    return response.strip()

### Step 2: Test your prompts


Run the following code to verify that your prompt produces the expected output:


In [12]:
# Unit test:
restaurant_paragraph = restaurant_list[0]
base_system_msg, base_user_prompt = restaurant_data_structure_prompt_generation(restaurant_paragraph=restaurant_paragraph)

test_response = llm_model(system_msg=base_system_msg, prompt_txt=base_user_prompt)
test_response=clean_json_response(test_response)
print(test_response)

{
    "name": "The Gilded Artichoke",
    "location": "Silver Lake",
    "type": "upscale bistro",
    "food_style": "Farm-to-Table Californian",
    "rating": 4.5,
    "price_range": 5,
    "signatures": [],
    "vibe": "bohemian chic energy",
    "environment": "high-end greenhouse with reclaimed wood and floor-to-ceiling windows",
    "shortcomings": []
}


### Step 3: Validate the LLM outputs


LLMs are not always perfectly reliable and may produce errors, even when strict rules are specified in the prompt. To ensure that the generated outputs strictly conform to the required JSON format, you will learn how to define a function that formally validates the LLM output. 

Here, you are provided with the function and the test. You don't have to implement anything. Run the following code to validate the test response you obtained in Step 2.


In [13]:
# Validation
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional

### 3.1. Define the schema
class Restaurant(BaseModel):
    name: str
    location: str
    type: str
    food_style: str
    rating: Optional[float] = None
    price_range: Optional[int] = None
    signatures: List[str] = Field(default_factory=list)
    vibe: Optional[str] = None
    environment: str
    shortcomings: List[str] = Field(default_factory=list)


### 3.2. Use the validation method to validate the test_response from the unit test
try:
    restaurant_data = Restaurant.model_validate_json(test_response)
    print(f"Success! Validated: {restaurant_data.name}")
except ValidationError as e:
    print(f"Validation failed: {e.json()}")

Success! Validated: The Gilded Artichoke


Don't worry if your validation failed; we'll address that in the next exercise. 


----


## **Exercise 4: Structure all the restaurant data**


### Step 1: Define prompts to have an LLM that auto repairs outputs to JSON format


As mentioned earlier, LLMs are not perfect. While you previously defined a schema to validate the output, validation alone does not fix incorrect results. To address this, introduce an additional LLM “expert” whose sole responsibility is to automatically repair and correct outputs that do not conform to the required JSON format.

In this step, you will define the prompt template function for this LLM. This function takes:
- `candidate_json_output`: The candidate json output
- `error_message`: The error message generated by the schema validation if a mistake is detected

In your prompts, you should tell LLM:
- The original wrong output by feeding `candidate_json_output`
- The guidance on correction by feeding `error_message`


In [14]:
def JSON_auto_repair_prompts(candidate_json_output, error_message):

    auto_repair_system_msg = """
You are a JSON repair assistant.

Your task is to repair an invalid JSON response so that it matches the
required restaurant data schema.

Rules:
1. Return ONLY valid JSON.
2. Do NOT use Markdown.
3. Do NOT use ```json or ``` code fences.
4. Do NOT add explanations or comments.
5. Preserve all correct information from the original output.
6. Fix only the problems identified by the validation error.
7. Do not invent or add information that is not present in the original output.
8. The final response must be a single valid JSON object.
9. The JSON must match the expected restaurant schema.
"""

    auto_repair_prompt = f"""
The following candidate JSON output failed validation.

Original candidate output:
{candidate_json_output}

Validation error:
{error_message}

Task:
Repair the candidate JSON according to the validation error.

Return ONLY the corrected JSON object.
Do not include ```json, Markdown, explanations, or any additional text.
"""

    return auto_repair_system_msg, auto_repair_prompt

### Step 2: Run the for loop to go over all the restaurant data in the list (approximately 20 minutes)


You will structure each restaurant description paragraph into a JSON-formatted output by iterating through the dataset using a for loop.

Although using an LLM to auto repair the response format is not the best practice, here you are guaranteed that the generation and repair tasks here are simple enough for LLMs to accomplish.


In [15]:
restaurant_paragraph

'**The Gilded Artichoke** brings a **bohemian chic** energy to the hills of **Silver Lake**, operating as an **upscale bistro** that prioritizes **Farm-to-Table Californian** ingredients. The space feels like a high-end greenhouse with its reclaimed wood and floor-to-ceiling windows, perfectly complementing the **4.5/5** rating earned by its lavender-rubbed roasted chicken and delicate heirloom tomato tarts.  Price range: $$$$'

In [16]:
import json

structured_restaurant_lists = []

for i, restaurant_paragraph in enumerate(restaurant_list):

    ### 2.1: Produce your initial output
    base_system_msg, base_user_prompt = restaurant_data_structure_prompt_generation(
        restaurant_paragraph
    )

    candidate_json_output = llm_model(
        system_msg=base_system_msg,
        prompt_txt=base_user_prompt
    )

    candidate_json_output = clean_json_response(candidate_json_output)

    ### 2.2: Validation and Auto Correction
    max_retries = 3

    for attempt in range(max_retries):

        try:
            # Validate JSON
            restaurant_data = Restaurant.model_validate_json(
                candidate_json_output
            )
            print(f"Restaurant {i} validated successfully: {restaurant_data.name}")

            # Valid -> stop retrying
            break

        except ValidationError as e:

            # If this was the last attempt, skip this restaurant
            if attempt == max_retries - 1:
                print(
                    f"Validation failed for restaurant {i + 1} "
                    f"after {max_retries} attempts."
                )
                restaurant_data = None
                break

            # Get validation error
            error_message = e.json()

            # Create repair prompts
            auto_repair_system_msg, auto_repair_prompt = JSON_auto_repair_prompts(
                candidate_json_output=candidate_json_output,
                error_message=error_message
            )

            # Ask LLM to repair
            candidate_json_output = llm_model(
                system_msg=auto_repair_system_msg,
                prompt_txt=auto_repair_prompt
            )

            # Clean repaired response
            candidate_json_output = clean_json_response(
                candidate_json_output
            )

    ### 2.3: Append only successfully validated restaurants
    if restaurant_data is not None:
        structured_restaurant_lists.append(
            restaurant_data.model_dump()
        )

    # Progress
    if (i + 1) % 20 == 0:
        print(f'{i + 1} out of {len(restaurant_list)} is done')


print('ALL DONE!!')
print(f'Successfully processed: {len(structured_restaurant_lists)}')

Restaurant 0 validated successfully: The Gilded Artichoke
Restaurant 1 validated successfully: Mar de Cortez
Validation failed for restaurant 3 after 3 attempts.
Validation failed for restaurant 4 after 3 attempts.
Validation failed for restaurant 5 after 3 attempts.
Validation failed for restaurant 6 after 3 attempts.
Restaurant 6 validated successfully: Velvet & Vine
Restaurant 7 validated successfully: The Neon Noodle Bar
Restaurant 8 validated successfully: The Morning Echo
Validation failed for restaurant 10 after 3 attempts.
Validation failed for restaurant 11 after 3 attempts.
Restaurant 11 validated successfully: Salt & Sand
Restaurant 12 validated successfully: Guerrero Kitchen
Validation failed for restaurant 14 after 3 attempts.
Validation failed for restaurant 15 after 3 attempts.
Restaurant 15 validated successfully: The Blue Fig
Restaurant 16 validated successfully: The Vinyl Bird
Restaurant 17 validated successfully: The Obsidian Room
Validation failed for restaurant 19 

Take a screenshot of the Python code, clearly showing your implementation and the output from Step 2. Name the screenshot `M1L1_structure_for_loop.jpg`. 


### Step 3: Save the list to a JSON file


You are almost there! First, print the 50th item in your structured_restaurant_lists.


In [17]:
### Your Code Here:
### Print the 50th item in the structured_restaurant_lists

print(structured_restaurant_lists[49])


{'name': 'Pico Provisions', 'location': 'Picfair Village', 'type': 'charming and sun-soaked corner cafe', 'food_style': 'New American', 'rating': 4.3, 'price_range': 3, 'signatures': [], 'vibe': 'relaxed, neighborhood-centric', 'environment': 'small, dog-friendly patio', 'shortcomings': []}


In [21]:
structured_restaurant_lists_json

[{'name': 'The Gilded Artichoke',
  'location': 'Silver Lake',
  'type': 'upscale bistro',
  'food_style': 'Farm-to-Table Californian',
  'rating': 4.5,
  'price_range': 5,
  'signatures': [],
  'vibe': 'bohemian chic energy',
  'environment': 'high-end greenhouse with reclaimed wood and floor-to-ceiling windows',
  'shortcomings': [],
  'itemId': 1000001},
 {'name': 'Mar de Cortez',
  'location': 'Santa Monica',
  'type': 'casual taqueria',
  'food_style': 'Baja-style seafood',
  'rating': 4.2,
  'price_range': 1,
  'signatures': ['beer-battered snapper tacos', 'zesty octopus ceviche'],
  'vibe': 'salt-air energy',
  'environment': 'a premier sun-drenched spot for open-air dining near the pier.',
  'shortcomings': [],
  'itemId': 1000002},
 {'name': 'Velvet & Vine',
  'location': 'Pasadena',
  'type': 'romantic wine bar',
  'food_style': 'French-Mediterranean',
  'rating': 4.1,
  'price_range': 2,
  'signatures': ['duck confit sliders'],
  'vibe': 'sophisticated backdrop for evening d

Then, save your processed data to your folder by running the following code!


In [20]:
import json

structured_restaurant_lists_json = structured_restaurant_lists

# Assign itemId to each restaurant
for i, response in enumerate(structured_restaurant_lists_json):
    response["itemId"] = 1000001 + i

# Save to ../data/
filename = "../data/structured_restaurant_data.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump(
        structured_restaurant_lists_json,
        f,
        indent=4,
        ensure_ascii=False
    )

print(f"Saved {len(structured_restaurant_lists_json)} restaurants to {filename}")

Saved 96 restaurants to ../data/structured_restaurant_data.json


You can download the saved JSON file to your local folder. Be sure to keep it in a safe location, as you will need it for future assignments.


----


## **Conclusion**


You have successfully applied GenAI tools to transform the unstructured text data into a well-structured JSON file!


## Authors


[Jianping (Mike) Ye](https://www.linkedin.com/in/jianping-ye/)


<!--## Changelog
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2026-02-02|1|Jianping Ye|Create lab|
|2026-02-09|2|Jojy John|ID Review|-->


Copyright IBM Corporation. All rights reserved.
